# **Stage 3: Graph-guided Expansion**

In [55]:
import json
import networkx as nx
import pickle

## Load data and graph

In [56]:
with open("./data/filtered_chunks.json", 'r', encoding='utf-8') as f:
    chunked_data = json.load(f)

with open("./data/grounded_triplets.json", 'r', encoding='utf-8') as f:
    grounded_triplets = json.load(f)

with open("./data/semantic_chunks.json", 'r', encoding='utf-8') as f:
    semantic_chunks = json.load(f)

with open('./data/triplets.gpickle', 'rb') as f:
    G = pickle.load(f)

In [57]:
Dq = [chunk['chunk_id'] for chunk in semantic_chunks]

## Get relevant subgraph ${G_q}^0$ of $D_q$

In [58]:
dq_ids = {c["chunk_id"] for c in semantic_chunks} if "semantic_chunks" in globals() else set()

Gq0_edges = [
    (u, v, data) for u, v, data in G.edges(data=True)
    if data.get("source_chunk_id") in dq_ids
]

Gq0 = nx.MultiDiGraph()
Gq0.add_edges_from(Gq0_edges);

## Traverse $m$-hop neighborhood of $G_q$ and get the expanded subgraph ${G_q}^m$

In [59]:
m = 1   # number of hops
visited = set(Gq0.nodes())

for _ in range(m):
    new_nodes = set()
    for node in list(visited):
        neighbors = list(G.neighbors(node))
        new_nodes.update(neighbors)
    visited.update(new_nodes)

Gqm = G.subgraph(visited).copy()

## Get expanded chunks ${D_q}^m$

In [60]:
expanded_chunks = set()

for u, v, data in Gqm.edges(data=True):
    if "source_chunk_id" in data:
        expanded_chunks.add(data["source_chunk_id"])

print("Number of expanded chunk IDs:", len(expanded_chunks))
print("Sample expanded chunk IDs:", list(expanded_chunks)[:10])

Number of expanded chunk IDs: 28
Sample expanded chunk IDs: ['5ac3165c5542995ef918c10a_doc5_chunk0', '5a8b57f25542995d1e6f1371_doc7_chunk0', '5a7320565542991f9a20c61d_doc8_chunk0', '5abc0a5d5542993f40c73c64_doc9_chunk1', '5ae2b770554299495565db0f_doc6_chunk0', '5a877e5d5542993e715abf7d_doc7_chunk0', '5abf63f15542997ec76fd3ea_doc5_chunk0', '5a8a3e745542996c9b8d5e70_doc1_chunk0', '5abbf698554299114383a0b5_doc3_chunk1', '5a8a3e745542996c9b8d5e70_doc4_chunk0']


In [61]:
chunks_dict = {
    c["chunk_id"]: c["chunk_text"]
    for c in chunked_data
    if isinstance(c, dict) and "chunk_id" in c
}

expanded_chunk_texts = {
    cid: chunks_dict[cid]
    for cid in expanded_chunks
    if cid in chunks_dict
}


In [62]:
expanded_chunk_texts

{'5ac3165c5542995ef918c10a_doc5_chunk0': 'John Samuel Waters Jr. (born April 22, 1946) is an American film director, screenwriter, author, actor, stand-up comedian, journalist, visual artist, and art collector, who rose to fame in the early 1970s for his transgressive cult films.',
 '5a8b57f25542995d1e6f1371_doc7_chunk0': 'Sinister is a 2012 supernatural horror film directed by Scott Derrickson and written by Derrickson and C. Robert Cargill.  It stars Ethan Hawke as fictional true-crime writer Ellison Oswalt who discovers a box of home movies in his attic that puts his family in danger.',
 '5a7320565542991f9a20c61d_doc8_chunk0': 'The 1989 season was the Houston Oilers 30th season and their 20th in the National Football League (NFL).  The franchise scored 365 points while the defense gave up 412 points.  Their record of 9 wins and 7 losses resulted in a second-place finish in the AFC Central Division.  The Oilers appeared once on Monday Night Football and appeared in the playoffs for t

## Handle output

In [63]:
import json
import re

chunk_lookup = {c["chunk_id"]: c for c in chunked_data}
expanded_chunk_ids = set()

for u, v, data in Gqm.edges(data=True):
    for key in ["source_chunk_id", "target_chunk_id"]:
        if key in data and data[key] in chunk_lookup:
            expanded_chunk_ids.add(data[key])

for node in Gqm.nodes():
    if node in chunk_lookup:
        expanded_chunk_ids.add(node)

expanded_chunks_out = []
for cid in expanded_chunk_ids:
    c = chunk_lookup[cid]
    m = re.search(r"_doc(\d+)_chunk(\d+)", cid)
    doc_index = int(m.group(1)) if m else None
    chunk_index = int(m.group(2)) if m else None
    expanded_chunks_out.append({
        "chunk_id": cid,
        "source_doc_title": c.get("source_doc_title", ""),
        "chunk_text": c.get("chunk_text", ""),
        "question_id": cid.split("_doc")[0],
        "doc_index": doc_index,
        "chunk_index": chunk_index
    })

with open("./data/expanded_chunks.json", "w", encoding="utf-8") as f:
    json.dump(expanded_chunks_out, f, ensure_ascii=False, indent=4)

with open("./data/umq.pickle", "wb") as f:
    pickle.dump(Gqm, f)
